# Predicting `arrears_next_month`
Follows `protocol.yaml`: temporal holdout from `holdout.csv`, walk-forward CV (5 splits) on training rows only, selection by mean CV ROC AUC, one scoring of the chosen model on the holdout.

**Feature decisions.** `meter_ref` (entity key) and `billing_month` (time, split only) are excluded per the protocol. `final_notice_sent` is also excluded: per `columns.csv` it is recorded in the billing cycle *after* the billing month, i.e. after prediction time, and is a consequence of arrears (leakage).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold = np.zeros(len(df), bool); is_hold[hold_idx]=True
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
train_pos = np.flatnonzero(~is_hold); hold_pos = np.flatnonzero(is_hold)
train = df.iloc[train_pos]; hold = df.iloc[hold_pos]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min(), train[TARGET].mean())

## Candidates (named before any scoring)
Preprocessing lives inside each pipeline, so it is fitted within each CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('num', StandardScaler() if scale else 'passthrough', NUM)])
candidates = {
 'baseline_prior': make_pipeline(prep(False), DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.1': make_pipeline(prep(True), LogisticRegression(C=0.1, max_iter=1000)),
 'random_forest': make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=0)),
 'gbm_depth2': make_pipeline(prep(False), GradientBoostingClassifier(max_depth=2, n_estimators=150, learning_rate=0.05, random_state=0)),
}
print(list(candidates))

## Walk-forward CV on training rows
Training months are split into 6 consecutive blocks; fold k trains on blocks before k+1 and validates on block k+1.

In [ ]:
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months, 6)
folds=[]
for k in range(1,6):
    tr = np.flatnonzero(train.billing_month.isin(np.concatenate(blocks[:k])).to_numpy())
    va = np.flatnonzero(train.billing_month.isin(blocks[k]).to_numpy())
    folds.append((train_pos[tr], train_pos[va]))
for tr,va in folds: print(len(tr), len(va), df.billing_month.iloc[tr].max().date(), df.billing_month.iloc[va].min().date())
X=df[FEATURES]; y=df[TARGET]
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(X.iloc[tr], y.iloc[tr])
        s.append(float(roc_auc_score(y.iloc[va], mm.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame(cv_scores).T; cv_table['mean']=cv_table.mean(axis=1)
cv_table.sort_values('mean',ascending=False)

## Choose by mean CV ROC AUC, refit on all training rows, score once on holdout

In [ ]:
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
final = clone(candidates[chosen]).fit(X.iloc[train_pos], y.iloc[train_pos])
holdout_auc = float(roc_auc_score(y.iloc[hold_pos], final.predict_proba(X.iloc[hold_pos])[:,1]))
print(chosen, np.mean(cv_scores[chosen]), holdout_auc)

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[tr.tolist(), va.tolist()] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': hold_pos.tolist(),
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The headline is the chosen model's holdout ROC AUC printed above. `final_notice_sent` would likely inflate performance but is unavailable at prediction time, so it is not used.